In [1]:
!pip install -q -U langchain-text-splitters chromadb sentence-transformers

from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions

documents = {
    "ai_overview.txt": """Artificial Intelligence, or AI, is the field of computer science focused on building systems that can perform tasks which normally require human intelligence. These tasks include understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following hardcoded rules. Instead of programming every decision by hand, we feed the system many examples, and it learns the underlying pattern on its own.

Deep Learning takes Machine Learning further by using neural networks with many layers. These layered networks can automatically discover complex patterns in large amounts of data, such as recognizing faces in photos or understanding the meaning of a sentence.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a language model. Instead of relying only on what a language model memorized during training, RAG first retrieves relevant information from a document collection, then passes that information to the model so it can generate a more accurate, grounded answer.""",

    "embeddings_in_practice.txt": """Modern search engines increasingly rely on embeddings rather than simple keyword matching. An embedding model converts a search query into the same vector space as the stored documents, allowing the system to find results that match the meaning of the query, not just its exact words.

This is especially powerful for customer support systems, where a user might phrase a question very differently from how it appears in a help article, yet still expect to find the right answer.

Vector databases such as ChromaDB store these embeddings and allow fast similarity search across thousands or millions of documents at once."""
}

splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""]
)

client = chromadb.Client()

embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.create_collection(
    name="rag_without_api",
    embedding_function=embedding_function
)

for source, text in documents.items():
    chunks = splitter.split_text(text)

    collection.add(
        documents=chunks,
        metadatas=[
            {
                "source": source,
                "chunk_index": i
            }
            for i in range(len(chunks))
        ],
        ids=[
            f"{source}_chunk_{i}"
            for i in range(len(chunks))
        ]
    )

print("Total chunks stored:", collection.count())


def retrieve_context(query, n_results=4, threshold=0.25):

    results = collection.query(
        query_texts=[query],
        n_results=min(n_results, collection.count()),
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    retrieved = []

    for document, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):

        similarity = 1 - distance

        if similarity >= threshold:

            retrieved.append({
                "text": document,
                "source": metadata["source"],
                "chunk_index": metadata["chunk_index"],
                "similarity": round(similarity, 4)
            })

    return retrieved


def generate_local_answer(query):

    context = retrieve_context(query)

    if not context:
        return "I don't know based on the provided documents."

    answer_parts = []

    for item in context[:3]:
        answer_parts.append(item["text"])

    answer = " ".join(answer_parts)

    sources = sorted(
        set(
            item["source"]
            for item in context
        )
    )

    return (
        answer
        + "\n\nSources: "
        + ", ".join(sources)
    )


question = "What is RAG and how does it work?"

retrieved_chunks = retrieve_context(question)

print("\n" + "=" * 60)
print("STEP 4: RAG ANSWERING WITHOUT API KEY")
print("=" * 60)

print("\nQuestion:")
print(question)

print("\nRetrieved Context:")

for item in retrieved_chunks:
    print(
        "\nSource:",
        item["source"],
        "| Chunk:",
        item["chunk_index"],
        "| Similarity:",
        item["similarity"]
    )
    print(item["text"])

print("\nGenerated Answer:")
print(generate_local_answer(question))


question2 = "What is deep learning?"

print("\n" + "=" * 60)
print("TEST 2")
print("=" * 60)

print("\nQuestion:")
print(question2)

print("\nAnswer:")
print(generate_local_answer(question2))


question3 = "What is the best recipe for chocolate cake?"

print("\n" + "=" * 60)
print("TEST 3: OUT-OF-SCOPE QUESTION")
print("=" * 60)

print("\nQuestion:")
print(question3)

print("\nAnswer:")
print(generate_local_answer(question3))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total chunks stored: 11

STEP 4: RAG ANSWERING WITHOUT API KEY

Question:
What is RAG and how does it work?

Retrieved Context:

Source: ai_overview.txt | Chunk: 5 | Similarity: 0.4115
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a language model

Source: ai_overview.txt | Chunk: 6 | Similarity: 0.3741
. Instead of relying only on what a language model memorized during training, RAG first retrieves relevant information from a document collection, then passes that information to the model so it can generate a more accurate, grounded answer.

Generated Answer:
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a language model . Instead of relying only on what a language model memorized during training, RAG first retrieves relevant information from a document collection, then passes that information to the model so it can generate a more accurate, grounded answer.

Sources: ai_overview.txt

TEST 2

Questi